In [1]:
# ── Cell 1: Setup ──
import sys
sys.path.append("..")  # Add project root to path

from config.settings import *
from config.grid_config import get_active_region
from src.data_collection.grid_generator import GridGenerator
from src.data_collection.land_mask import LandMasker

print("✅ Imports successful")
print(f"Active region: {get_active_region()['name']}")

2026-05-04 11:08:14 | INFO     | io | 📁 Storage format: parquet (parquet available: True)


✅ Imports successful
Active region: India


In [2]:
# ── Cell 2: Generate Grid ──
generator = GridGenerator()
grid_df = generator.generate()
print(generator.summary())
grid_df.head(10)

2026-05-04 11:08:19 | INFO     | grid_generator | 🌍 GridGenerator initialized for: India | Spacing: 0.5°
2026-05-04 11:08:19 | INFO     | grid_generator | 📍 Generated 3540 grid points | Lat range: [8.0, 37.0] | Lon range: [68.0, 97.5]


{'region': 'India', 'total_points': 3540, 'grid_spacing': 0.5, 'lat_range': (np.float64(8.0), np.float64(37.0)), 'lon_range': (np.float64(68.0), np.float64(97.5)), 'approx_resolution_km': 55.5}


,point_id,latitude,longitude
0,PT_00000,8.0,68.0
1,PT_00001,8.0,68.5
2,PT_00002,8.0,69.0
3,PT_00003,8.0,69.5
4,PT_00004,8.0,70.0
5,PT_00005,8.0,70.5
6,PT_00006,8.0,71.0
7,PT_00007,8.0,71.5
8,PT_00008,8.0,72.0
9,PT_00009,8.0,72.5


In [3]:
# ── Cell 3: Apply Land Mask ──
masker = LandMasker()
land_df = masker.apply(grid_df)
masker.save_land_points(land_df)
land_df.head(10)

2026-05-04 11:08:26 | INFO     | land_mask | 🗺️  Loaded boundary from: india_boundary.geojson
2026-05-04 11:08:29 | INFO     | land_mask | 🏝️  Land masking: 3540 → 1153 points (2387 ocean/border points removed)
2026-05-04 11:08:29 | INFO     | land_mask | 💾 Saved land points: D:\Semester 6\Unsuperwised Learning\Project\weather-pattern-clustering\data\interim\grid_points_india.csv


,point_id,latitude,longitude
0,PT_00000,8.5,77.0
1,PT_00001,8.5,77.5
2,PT_00002,8.5,78.0
3,PT_00003,9.0,77.0
4,PT_00004,9.0,77.5
5,PT_00005,9.0,78.0
6,PT_00006,9.5,76.5
7,PT_00007,9.5,77.0
8,PT_00008,9.5,77.5
9,PT_00009,9.5,78.0


In [4]:
# ── Cell 4: Visualize Grid Points on Map ──
import folium

region = get_active_region()
center_lat = (region["lat_min"] + region["lat_max"]) / 2
center_lon = (region["lon_min"] + region["lon_max"]) / 2

m = folium.Map(location=[center_lat, center_lon], zoom_start=5)

# Plot land points
for _, row in land_df.iterrows():
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=2,
        color="green",
        fill=True,
        fillOpacity=0.7,
        popup=f"{row['point_id']}: ({row['latitude']}, {row['longitude']})",
    ).add_to(m)

m.save(str(FIGURES_DIR / "india_grid_points.html"))
print(f"✅ Map saved! Total land points: {len(land_df)}")
m

✅ Map saved! Total land points: 1153
